# Mean field of the faculae (sweep_facula_B.dag)

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sunsim import observations as obs

BASE_FB = '/data/spotless/scratch/sstucki/starsim/facula_B'
TAG = 'facula_B_m45s_size_4096_N60'
SETS = os.path.join(BASE_FB, 'sets_' + TAG)
START = '2015-07-18'                             # t = 0 of the simulations
OBS = '/data/spotless/scratch/sstucki/starsim/observations'
HARPSN = os.path.join(OBS, 'harpn_sun_release_timeseries_2015-2025.csv')
INTERVENTIONS = os.path.join(OBS, 'harpsn_intervention_days.txt')
SORCE, TSIS = os.path.join(OBS, 'DataSSI - SORCE.sav'), os.path.join(OBS, 'tsis_tsi_24hr.csv')
MAX_FF_FC, MAX_FF_SP = 20.0, 5.0                 # epochs left out (bad HMI images), as obs.sweep_stats
PARAMS = ['noise_th', 'B_th', 'I_th', 'A_th', 'B_fc', 'T_sp']

METRICS = [('rv_rms', 'RV - HARPS-N', 'm/s'), ('fwhm_rms', 'FWHM - HARPS-N', 'm/s'), ('bis_rms', 'BIS - HARPS-N', 'm/s'),
           ('sorce_rms', 'TSI - SORCE/TIM', 'ppm'), ('tsis_rms', 'TSI - TSIS-1/TIM', 'ppm')]

df = obs.load_sweep_stats(SETS).sort_values(['I_th', 'A_th', 'B_fc']).reset_index(drop=True)
METRICS = [m for m in METRICS if m[0] in df and df[m[0]].notna().any()]
B_FC = sorted(df['B_fc'].unique())
THR = list(df[['I_th', 'A_th']].drop_duplicates().itertuples(index=False, name=None))   # (I_th, A_th) pairs
print(len(df), 'sets, B_fc =', ', '.join('%g' % b for b in B_FC))
print('(I_th, A_th) =', ', '.join('(%g, %g)' % t for t in THR))
df[PARAMS[2:5] + [m[0] for m in METRICS] + [m[0].replace('_rms', '_mad') for m in METRICS]].round(3)

In [ ]:
plt.rcParams.update({
    'font.size': 14,
    'axes.labelsize': 16,
    'axes.titlesize': 18,
    'xtick.labelsize': 14,
    'ytick.labelsize': 14,
    'legend.fontsize': 12,
    'figure.titlesize': 18,
})
try:
    import cmasher as cmr
    CMAP = cmr.get_sub_cmap(cmr.torch, 0, 0.95)
except ImportError:
    CMAP = 'inferno'

# one colour per A_th (from cmr.torch, black and white ends left out), one line style / marker per I_th
A_TH, I_TH = sorted(df['A_th'].unique()), sorted(df['I_th'].unique())
A_COLOURS = dict(zip(A_TH, plt.get_cmap(CMAP)(np.linspace(0.15, 0.85, len(A_TH)))))
I_STYLES = dict(zip(I_TH, [('-', 'o'), ('--', 's'), (':', '^'), ('-.', 'D')]))


def thr_label(i_th, a_th):
    return r'$I_{th}$ = %g, $A_{th}$ = %g' % (i_th, a_th)


def grid(df, key):
    """Values of key with the (I_th, A_th) pairs as rows and B_fc as columns."""
    g = df.pivot_table(index=['I_th', 'A_th'], columns='B_fc', values=key, aggfunc='first')
    return g.reindex(index=pd.MultiIndex.from_tuples(THR, names=['I_th', 'A_th']), columns=B_FC).values

## Heatmaps of the sweep

In [ ]:
def sweep_heatmaps(df, metrics, kind='rms', save=None):
    """One heatmap per metric: (I_th, A_th) pairs (rows) against B_fc (columns), coloured by the value over the
    minimum of the panel; the values in the cells, the best set of every panel outlined in red."""
    n = len(metrics)
    fig, axes = plt.subplots(n, 1, figsize=(1.3 * len(B_FC) + 4, (0.55 * len(THR) + 1.1) * n), sharex=True,
                             squeeze=False)
    for ax, (k, title, unit) in zip(axes[:, 0], metrics):
        vals = grid(df, k.replace('_rms', '_' + kind))
        rel = vals / np.nanmin(vals)
        im = ax.imshow(rel, aspect='auto', cmap=CMAP, vmin=1, vmax=np.nanmax(rel))
        for i in range(len(THR)):
            for j in range(len(B_FC)):
                if np.isfinite(vals[i, j]):
                    ax.text(j, i, '%.3g' % vals[i, j], ha='center', va='center', fontsize=11,
                            color='w' if im.norm(rel[i, j]) < 0.6 else 'k')
        i, j = np.unravel_index(np.nanargmin(vals), vals.shape)
        ax.add_patch(plt.Rectangle((j - 0.5, i - 0.5), 1, 1, fill=False, ec='red', lw=3))
        ax.set_yticks(range(len(THR)))
        ax.set_yticklabels(['%g / %g' % t for t in THR], fontsize=12)
        ax.set_ylabel(r'$I_{th}$ / $A_{th}$', fontsize=13)
        ax.set_title('%s %s [%s]' % (title, kind.upper(), unit), fontsize=15)
        cbar = fig.colorbar(im, ax=ax, fraction=0.03, pad=0.02)
        cbar.set_label('/ min', fontsize=12)
    ax = axes[-1, 0]
    ax.set_xticks(range(len(B_FC)))
    ax.set_xticklabels(['%g' % b for b in B_FC])
    ax.set_xlabel(r'$B_{fc}$ [G]')
    fig.tight_layout()
    if save:
        fig.savefig(save, bbox_inches='tight')
    return fig, axes


sweep_heatmaps(df, METRICS, 'rms')
plt.show()
sweep_heatmaps(df, METRICS, 'mad')
plt.show()

## RMS against the mean field of the faculae

In [ ]:
def sweep_curves(df, metrics, kind='rms', save=None):
    """value of every metric against B_fc, one curve per (I_th, A_th) pair (colour: A_th, style: I_th),
    the best set of every panel marked by a red circle."""
    fig, axes = plt.subplots(1, len(metrics), figsize=(4.6 * len(metrics), 4.4), sharex=True, squeeze=False)
    for ax, (k, title, unit) in zip(axes[0], metrics):
        key = k.replace('_rms', '_' + kind)
        for i_th, a_th in THR:
            d = df[(df['I_th'] == i_th) & (df['A_th'] == a_th)]
            ls, mk = I_STYLES[i_th]
            ax.plot(d['B_fc'], d[key], ls=ls, marker=mk, ms=8, lw=2, c=A_COLOURS[a_th], label=thr_label(i_th, a_th))
        b = df.loc[df[key].idxmin()]
        ax.plot(b['B_fc'], b[key], 'o', ms=16, mfc='none', mec='red', mew=2.5)
        ax.axvline(200, c='0.7', lw=1, ls=':')
        ax.grid(alpha=0.25)
        ax.set_title(title, fontsize=15)
        ax.set_xlabel(r'$B_{fc}$ [G]')
        ax.set_ylabel('%s [%s]' % (kind.upper(), unit))
    axes[0, 0].legend(fontsize=10)
    fig.tight_layout()
    if save:
        fig.savefig(save, bbox_inches='tight')
    return fig, axes


sweep_curves(df, METRICS, 'rms')
plt.show()
sweep_curves(df, METRICS, 'mad')
plt.show()

## Best parameter set

In [ ]:
# score of a set: mean over the metrics of its RMS over the smallest RMS of that metric (1 = best in every metric)
REL = pd.DataFrame({k: df[k] / df[k].min() for k, _, _ in METRICS})
df['score'] = REL.mean(axis=1)
df['score_rv_tsis'] = REL[[k for k in ('rv_rms', 'tsis_rms') if k in REL]].mean(axis=1)

print('best set of every metric:')
for k, title, unit in METRICS:
    b = df.loc[df[k].idxmin()]
    print('  %-17s I_th = %g, A_th = %g, B_fc = %g G  ->  %.4g %s' % (title, b['I_th'], b['A_th'], b['B_fc'], b[k], unit))

for s, what in (('score', 'all metrics'), ('score_rv_tsis', 'RV and TSIS-1')):
    b = df.loc[df[s].idxmin()]
    print('\nBEST SET (%s): ' % what + ', '.join('%s = %g' % (p, b[p]) for p in PARAMS if p in b))
    print('  ' + ', '.join('%s %.4g' % (k.replace('_rms', ''), b[k]) for k, _, _ in METRICS) + ', score %.3f' % b[s])
    print('  folder:', b['folder'])

BEST = df.loc[df['score'].idxmin()]

fig, axes = plt.subplots(1, 2, figsize=(2 * (1.1 * len(B_FC) + 3), 0.6 * len(THR) + 2), sharey=True)
for ax, (s, what) in zip(axes, (('score', 'all metrics'), ('score_rv_tsis', 'RV and TSIS-1'))):
    vals = grid(df, s)
    im = ax.imshow(vals, aspect='auto', cmap=CMAP, vmin=1, vmax=np.nanmax(vals))
    for i in range(len(THR)):
        for j in range(len(B_FC)):
            if np.isfinite(vals[i, j]):
                ax.text(j, i, '%.2f' % vals[i, j], ha='center', va='center', fontsize=11,
                        color='w' if im.norm(vals[i, j]) < 0.6 else 'k')
    i, j = np.unravel_index(np.nanargmin(vals), vals.shape)
    ax.add_patch(plt.Rectangle((j - 0.5, i - 0.5), 1, 1, fill=False, ec='red', lw=3))
    ax.set_xticks(range(len(B_FC)))
    ax.set_xticklabels(['%g' % b for b in B_FC])
    ax.set_xlabel(r'$B_{fc}$ [G]')
    ax.set_title('Mean RMS / min (%s)' % what, fontsize=15)
    fig.colorbar(im, ax=ax, fraction=0.04, pad=0.02)
axes[0].set_yticks(range(len(THR)))
axes[0].set_yticklabels(['%g / %g' % t for t in THR], fontsize=12)
axes[0].set_ylabel(r'$I_{th}$ / $A_{th}$', fontsize=13)
fig.tight_layout()
plt.show()

## Time series

In [ ]:
def series(folder):
    """Time series of one set (obs._read_series), the bad epochs left out."""
    s = obs._read_series(os.path.join(SETS, folder))
    keep = ~((s['ff_fc'] > MAX_FF_FC) | (s['ff_sp'] > MAX_FF_SP))
    return {k: v[keep] for k, v in s.items()}


# every B_fc at the (I_th, A_th) of the best set
sub = df[(df['I_th'] == BEST['I_th']) & (df['A_th'] == BEST['A_th'])]
colours = plt.get_cmap('viridis')(np.linspace(0, 0.9, len(sub)))
fig, axes = plt.subplots(2, 1, figsize=(15, 8), sharex=True)
for c, (_, r) in zip(colours, sub.iterrows()):
    s = series(r['folder'])
    axes[0].plot(s['time'], 1e6 * (s['lc'] / np.nanmean(s['lc']) - 1), '.', ms=1.5, c=c,
                 label=r'$B_{fc}$ = %g G' % r['B_fc'])
    axes[1].plot(s['time'], s['rv'] - np.nanmedian(s['rv']), '.', ms=1.5, c=c)
axes[0].set_title(thr_label(BEST['I_th'], BEST['A_th']), fontsize=15)
axes[0].set_ylabel('Flux [ppm]')
axes[1].set_ylabel('RV [m/s]')
axes[1].set_xlabel('Time [days since %s]' % START)
axes[0].legend(markerscale=6, ncol=len(sub), fontsize=10)
fig.tight_layout()
plt.show()

## Best sets against the observations

In [ ]:
harps = obs.load_harpsn(HARPSN, START)
sorce, tsis = obs.load_sorce(SORCE, START), obs.load_tsis(TSIS, START)
cuts = obs.load_interventions(INTERVENTIONS, START)
for k in ('score', 'rv_rms', 'tsis_rms'):
    b = obs.best_sets(df, k, 1).iloc[0]
    print('best %s: I_th = %g, A_th = %g, B_fc = %g G (%.4g)' % (k, b['I_th'], b['A_th'], b['B_fc'], b[k]))
    s = series(b['folder'])
    ph = [obs.compare_photometry(sorce, s['time'], s['lc']), obs.compare_photometry(tsis, s['time'], s['lc'])]
    rv = obs.compare_rv(harps, s['time'], s['rv'], interventions=cuts)
    obs.print_summary(ph, rv)
    obs.plot_comparison(ph, rv, start=START)
    plt.show()